# NA²Q Training on Google Colab

Trains the NA²Q agent on the Directional Sensor Network environment.

**Tips:**
- Enable GPU: Runtime → Change runtime type → T4 GPU
- Mount Google Drive (Cell 2) to save checkpoints persistently across sessions
- Training auto-saves `best_model.pt` to `na2q/checkpoints/` (and to Drive if mounted)

In [ ]:
# ── 1. Clone repo ──────────────────────────────────────────────────────────────
import os
if not os.path.exists('DSN_NA2Q'):
    !git clone https://github.com/Chiseled141/DSN_NA2Q.git
%cd DSN_NA2Q

In [ ]:
# ── 2. (Optional) Mount Google Drive to persist checkpoints ───────────────────
# Skip this cell if you don't need persistent storage.
USE_DRIVE = True   # set False to skip
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/DSN_NA2Q/checkpoints'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    print(f'Drive mounted. Checkpoints will be mirrored to: {DRIVE_CHECKPOINT_DIR}')
else:
    print('Skipping Drive mount. Checkpoints only saved locally (lost on session end).')

In [ ]:
# ── 3. Install dependencies ────────────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Verify GPU ──────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Training config (edit here) ────────────────────────────────────────────
SCENARIO  = 1      # 1 = small (5 sensors, 6 targets)  ← fastest to train
                   # 2 = large (50 sensors, 60 targets)
                   # 3 = medium (15 sensors, 20 targets)

EPISODES  = None   # None = use default from config.py (50k/10k/20k per scenario)
                   # Set a number to override, e.g. 5000 for a quick run

RESUME    = False  # True = resume from existing checkpoint in na2q/checkpoints/

In [ ]:
# ── 6. (Optional) Restore checkpoint from Drive before training ────────────────
import shutil

LOCAL_CKPT_DIR = 'na2q/checkpoints'
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

if USE_DRIVE and RESUME:
    restored = []
    for fname in ['best_model.pt', 'final_model.pt', 'training_history.npz']:
        src = os.path.join(DRIVE_CHECKPOINT_DIR, fname)
        dst = os.path.join(LOCAL_CKPT_DIR, fname)
        if os.path.exists(src):
            shutil.copy(src, dst)
            restored.append(fname)
    if restored:
        print('Restored from Drive:', restored)
    else:
        print('No checkpoint found on Drive — starting fresh.')
        RESUME = False
elif RESUME:
    if not os.path.exists(os.path.join(LOCAL_CKPT_DIR, 'final_model.pt')):
        print('No local checkpoint found — starting fresh.')
        RESUME = False

In [ ]:
# ── 7. Run training ────────────────────────────────────────────────────────────
import sys

# Build CLI args (mirrors: python -m na2q.main --mode train --scenario X)
argv = ['na2q.main', '--mode', 'train', '--scenario', str(SCENARIO)]
if EPISODES is not None:
    argv += ['--episodes', str(EPISODES)]
if RESUME:
    argv += ['--resume']

sys.argv = argv

from na2q.main import main
main()

In [ ]:
# ── 8. Mirror checkpoints to Google Drive ─────────────────────────────────────
if USE_DRIVE:
    for fname in os.listdir(LOCAL_CKPT_DIR):
        src = os.path.join(LOCAL_CKPT_DIR, fname)
        dst = os.path.join(DRIVE_CHECKPOINT_DIR, fname)
        shutil.copy(src, dst)
    print('Checkpoints saved to Drive:', DRIVE_CHECKPOINT_DIR)
else:
    print('Drive not mounted — download manually below.')

In [ ]:
# ── 9. Download best model to your machine ─────────────────────────────────────
from google.colab import files

best = os.path.join(LOCAL_CKPT_DIR, 'best_model.pt')
history = os.path.join(LOCAL_CKPT_DIR, 'training_history.npz')

if os.path.exists(best):
    files.download(best)
if os.path.exists(history):
    files.download(history)

In [ ]:
# ── 10. (Optional) Quick eval after training ───────────────────────────────────
sys.argv = ['na2q.main', '--mode', 'test', '--scenario', str(SCENARIO),
            '--episodes', '10', '--verbose']

from na2q.main import run_test, parse_args
run_test(parse_args())